In [ ]:
"""
edit_coco_tracks.py
--------------------
Outil pour filtrer et renommer des tracks dans un fichier d'annotations COCO.
Les paramètres sont lus depuis un fichier galaxy_inputs/galaxy_inputs.json.

Structure attendue du galaxy_inputs.json :
{
    "input_json":   "annotations.json",
    "mode":         "keep",                            // "keep" ou "remove"
    "id":    "0,1,2",                           // track IDs à garder ou supprimer
    "rename":       "1:Aplysia_juvenile,2:Aplysia_adulte"  // optionnel, peut être "" ou absent
}
"""

import json
import sys
from pathlib import Path
from copy import deepcopy

# ── Lecture des inputs Galaxy ──────────────────────────────────────────────────
 
def load_galaxy_inputs(path: str = "galaxy_inputs/galaxy_inputs.json") -> dict:
    p = Path(path)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

        
def parse_track_ids(raw: str) -> set:
    """Parse '0,1,3' → {0, 1, 3}"""
    return {int(x.strip()) for x in raw.split(",") if x.strip() != ""}


def parse_renames(raw: str) -> dict:
    """Parse '0:Aplysia_adulte,1:Aplysia_juvenile' → {0: 'Aplysia_adulte', 1: 'Aplysia_juvenile'}"""
    result = {}
    for item in raw.split(","):
        item = item.strip()
        if not item:
            continue
        if ":" not in item:
            raise ValueError(f"Format invalide pour rename : '{item}' (attendu track_id:nom)")
        track_id_str, new_name = item.split(":", 1)
        result[int(track_id_str.strip())] = new_name.strip()
    return result


# ── Traitement principal ───────────────────────────────────────────────────────

def edit_coco(data: dict, mode: str, track_ids: set, renames: dict) -> dict:
    output = deepcopy(data)
    annotations = output.get("annotations", [])

    # ── 1. Info sur les tracks présents ───────────────────────────────────
    all_track_ids = {ann["track_id"] for ann in annotations}
    print(f"Track IDs présents dans le fichier : {sorted(all_track_ids)}")

    # ── 2. Filtrage ────────────────────────────────────────────────────────
    if mode == "keep":
        unknown = track_ids - all_track_ids
        if unknown:
            print(f"  Track IDs introuvables (keep) : {unknown}")
        annotations = [ann for ann in annotations if ann["track_id"] in track_ids]
        print(f" Tracks gardés : {sorted(track_ids & all_track_ids)}")

    elif mode == "remove":
        unknown = track_ids - all_track_ids
        if unknown:
            print(f"  Track IDs introuvables (remove) : {unknown}")
        annotations = [ann for ann in annotations if ann["track_id"] not in track_ids]
        print(f"  Tracks supprimés : {sorted(track_ids & all_track_ids)}")

    else:
        print(f" Mode inconnu : '{mode}'. Utilisez 'keep' ou 'remove'.", file=sys.stderr)
        sys.exit(1)

    # ── 3. Renommage des catégories ────────────────────────────────────────
    categories = output.get("categories", [])
    name_to_cat_id = {cat["name"]: cat["id"] for cat in categories}
    max_cat_id = max((cat["id"] for cat in categories), default=0)

    track_to_new_cat = {}
    for track_id, new_name in renames.items():
        if new_name not in name_to_cat_id:
            max_cat_id += 1
            categories.append({"id": max_cat_id, "name": new_name, "supercategory": ""})
            name_to_cat_id[new_name] = max_cat_id
            print(f" Nouvelle catégorie créée : id={max_cat_id}, name='{new_name}'")
        track_to_new_cat[track_id] = name_to_cat_id[new_name]

    for ann in annotations:
        if ann["track_id"] in track_to_new_cat:
            old_cat = ann["category_id"]
            ann["category_id"] = track_to_new_cat[ann["track_id"]]
            if old_cat != ann["category_id"]:
                print(
                    f"   Annotation id={ann['id']} (track {ann['track_id']}) : "
                    f"category_id {old_cat} → {ann['category_id']}"
                )

    # ── 4. Nettoyer les catégories non utilisées ───────────────────────────
    used_cat_ids = {ann["category_id"] for ann in annotations}
    categories = [cat for cat in categories if cat["id"] in used_cat_ids]

    # ── 5. Renuméroter les annotations ────────────────────────────────────
    for new_id, ann in enumerate(annotations, start=1):
        ann["id"] = new_id

    output["annotations"] = annotations
    output["categories"] = categories
    print(f"\n Résultat : {len(annotations)} annotation(s) conservée(s).")
    return output


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    # 1. Lire galaxy_inputs.json
    galaxy = load_galaxy_inputs("galaxy_inputs/galaxy_inputs.json")

    input_json    = galaxy["Annotation"][0]["path"]
    mode          = galaxy.get("mode", "").strip().lower()   # "keep" ou "remove"
    track_ids_raw = galaxy.get("id", "")              # ex: "0,1,2"
    rename_raw    = galaxy.get("rename", "")                 # ex: "1:Aplysia_juvenile"

    # 3. Parsing
    track_ids = parse_track_ids(track_ids_raw)
    renames   = parse_renames(rename_raw) if rename_raw.strip() else {}
    print(f"  path    : {input_json}")
    print(f"  Mode    : {mode}")
    print(f" Tracks  : {sorted(track_ids)}")
    if renames:
        print(f" Rename  : {renames}")

    # 4. Lecture du COCO JSON
    input_path = Path(input_json)
    if not input_path.exists():
        print(f" Fichier COCO introuvable : {input_path}", file=sys.stderr)
        sys.exit(1)

    with open(input_path, encoding="utf-8") as f:
        data = json.load(f)

    # 5. Traitement
    result = edit_coco(data, mode, track_ids, renames)

    # 6. Écriture
    input_path = Path(input_json)
    output_path = Path("outputs") / (input_path.stem + "_edited.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f" Fichier sauvegardé : {output_path}")


if __name__ == "__main__":
    main()